# 01. 데이터 품질 점검

`eda 예시.ipynb`의 순서인 **구조 이해 → 품질 진단 → 처리 결정 → 해석**을 PlaylistPro 데이터에 적용합니다.

> 분석 단위는 고객 1명 = 1행이며, Target은 `churned`(0=유지, 1=이탈)입니다. 관측 기준일과 결과 기간은 제공되지 않아 미래 30일 이탈로 해석하지 않습니다.

## 1. 데이터 로드와 분석 목적

- 원본 출처·라이선스·합성 판정 근거는 `docs/data_card.md`에서 관리합니다.
- 이 Notebook은 제출 패키지의 `data/processed/`에 복사된 최종 고객 단위 테이블을 읽습니다.
- 원본 값을 수정하지 않고 품질 문제를 계수합니다.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
train = pd.read_csv(ROOT / "data/processed/train.csv")
test = pd.read_csv(ROOT / "data/processed/test.csv")
print("train:", train.shape, "test:", test.shape)
train.head(3)

train: (125000, 20) test: (75000, 19)


,customer_id,age,location,subscription_type,payment_plan,num_subscription_pauses,payment_method,customer_service_inquiries,signup_date,weekly_hours,average_session_length,song_skip_rate,weekly_songs_played,weekly_unique_songs,num_favorite_artists,num_platform_friends,num_playlists_created,num_shared_playlists,notifications_clicked,churned
0,1,32,Montana,Free,Yearly,2,Paypal,Medium,-1606,22.391362,105.394516,0.176873,169,109,18,32,52,35,46,0
1,2,64,New Jersey,Free,Monthly,3,Paypal,Low,-2897,29.294210,52.501115,0.981811,55,163,44,33,12,25,37,1
2,3,51,Washington,Premium,Yearly,2,Credit Card,High,-348,15.400312,24.703696,0.048411,244,117,20,129,50,28,38,0


## 2. 데이터 구조 및 변수 이해

행·열 수, 자료형, 고객 키의 유일성, Train/Test 고객 교집합을 확인합니다. 식별자 `customer_id`는 모델 입력에서 제외합니다.

In [2]:
structure = pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(train), len(test)],
    "columns": [train.shape[1], test.shape[1]],
    "customer_id_unique": [train.customer_id.is_unique, test.customer_id.is_unique],
    "missing_cells": [int(train.isna().sum().sum()), int(test.isna().sum().sum())],
    "duplicate_rows": [int(train.duplicated().sum()), int(test.duplicated().sum())],
})
display(structure)
print("Train/Test customer_id 교집합:", len(set(train.customer_id) & set(test.customer_id)))
display(train.dtypes.rename("dtype").to_frame())

,dataset,rows,columns,customer_id_unique,missing_cells,duplicate_rows
0,train,125000,20,True,0,0
1,test,75000,19,True,0,0


Train/Test customer_id 교집합: 0


,dtype
customer_id,int64
age,int64
location,object
subscription_type,object
payment_plan,object
num_subscription_pauses,int64
payment_method,object
customer_service_inquiries,object
signup_date,int64
weekly_hours,float64


**해석:** 고객 키 중복·Train/Test 교집합·결측·완전 중복은 없습니다. 따라서 행 삭제나 임의 대체보다, Pipeline 안에서 미등록 범주와 향후 결측에 대비한 안전한 전처리를 유지하는 것이 적절합니다.

## 3. Target과 클래스 비율

Accuracy만으로 판단하지 않고 Recall, Precision, F1, PR-AUC를 함께 사용하는 이유를 확인합니다.

In [3]:
target = train["churned"].value_counts().sort_index().rename_axis("churned").to_frame("customers")
target["rate"] = target["customers"] / len(train)
display(target)

,customers,rate
churned,,
0,60826,0.486608
1,64174,0.513392


**해석:** 관측 이탈률은 약 51.3%로 극단적 불균형은 아닙니다. SMOTE를 기본 적용하지 않고, 동일 Fold의 PR-AUC와 Threshold별 오류량을 비교합니다.

## 4. 결측·중복·이상값과 데이터 계약 위반

통계적 극단값을 일괄 삭제하지 않고, 도메인상 불가능한 조합을 별도로 계수합니다.

In [4]:
quality = pd.DataFrame([
    {"check": "missing cells", "count": int(train.isna().sum().sum()), "action": "현재 없음; Pipeline 대치 유지"},
    {"check": "duplicate rows", "count": int(train.duplicated().sum()), "action": "삭제 없음"},
    {"check": "unique songs > played songs", "count": int((train.weekly_unique_songs > train.weekly_songs_played).sum()), "action": "수집 규칙 경고; 원본 보존"},
    {"check": "shared playlists > created", "count": int((train.num_shared_playlists > train.num_playlists_created).sum()), "action": "수집 규칙 경고; 원본 보존"},
])
quality["rate"] = quality["count"] / len(train)
display(quality)

,check,count,action,rate
0,missing cells,0,현재 없음; Pipeline 대치 유지,0.000000
1,duplicate rows,0,삭제 없음,0.000000
2,unique songs > played songs,36996,수집 규칙 경고; 원본 보존,0.295968
3,shared playlists > created,30778,수집 규칙 경고; 원본 보존,0.246224


**해석:** 고유 곡 수와 공유 플레이리스트에는 현실적으로 불가능한 조합이 각각 약 29.6%, 24.6% 존재합니다. 이를 임의 수정하면 생성 규칙을 새로 주입할 수 있으므로 원본을 보존하고 실제 서비스 적용 한계로 기록합니다.

## 5. 전처리 결정

| 항목 | 적용 | 근거 |
|---|---|---|
| 식별자 | `customer_id` 제외 | 일반화 불가능한 키 |
| 날짜 대용치 | `signup_date` → 경과일 | 실제 날짜가 아닌 음수 정수 |
| 결측 | 수치 median, 범주 최빈값 | 새 입력의 결측 대비 |
| 범주형 | One-Hot, unknown 허용 | 새 범주에서 추론 중단 방지 |
| 수치 변환 | `log_numeric` 채택 | 동일 Fold Logistic과 CatBoost 검증에서 근소하게 우수 |
| 불균형 | SMOTE 미적용 | 클래스 비율이 극단적이지 않고 Fold 내부 비교를 우선 |
| 누수 방지 | 모든 변환을 Pipeline/Fold 내부에서 fit | Validation 정보의 학습 유입 방지 |

## 6. 최종 요약

1. 고객 125,000명의 Train과 75,000명의 무라벨 Test를 사용합니다.
2. 결측·중복·고객 교집합은 없지만, 도메인상 불가능한 조합이 존재합니다.
3. 규칙 기반 합성으로 판단되므로 높은 점수를 실서비스 성능으로 일반화하지 않습니다.
4. 데이터 수정 대신 Train Fold 내부 Pipeline과 검증 경계를 유지합니다.
5. 상세 근거는 `reports/preprocessing_report.md`와 `docs/data_card.md`에 연결됩니다.